## **TRANSFORMATIONS**

In [0]:
# Read from the Bronze Delta Lake table
df_titles = (
    spark.read
    .format("delta")
    .load("abfss://bronze@mynetflixproject.dfs.core.windows.net/netflix_titles")
)


In [0]:
display(df_titles)

In [0]:
import pyspark.sql.functions as F

# 1. Check Row Counts
raw_df_titles = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("delimiter", ",")
    .csv("abfss://raw@mynetflixproject.dfs.core.windows.net/netflix_titles/")
)
raw_count = raw_df_titles.count()
bronze_count = df_titles.count()
count_difference = bronze_count - raw_count

# 2. Standardise schemas to strings for an exact text content comparison
raw_standardised = raw_df_titles.select([F.col(c).cast("string").alias(c) for c in raw_df_titles.columns])
bronze_standardised = df_titles.select([F.col(c).cast("string").alias(c) for c in raw_df_titles.columns])

# 3. Check for specific content differences
rows_in_raw_not_bronze = raw_standardised.subtract(bronze_standardised).count()
rows_in_bronze_not_raw = bronze_standardised.subtract(raw_standardised).count()

# 4. Print Complete Validation Summary
print("=" * 45)
print("🔍 COMPLETE RAW VS. BRONZE VERIFICATION")
print("=" * 45)
print(f"Raw Row Count:                {raw_count:,}")
print(f"Bronze Row Count:             {bronze_count:,}")
print(f"Row Count Difference:         {count_difference:,}")
print("-" * 45)
print(f"Rows in Raw but NOT Bronze:   {rows_in_raw_not_bronze}")
print(f"Rows in Bronze but NOT Raw:   {rows_in_bronze_not_raw}")
print("=" * 45)

# Final Verdict Check
counts_match = (raw_count == bronze_count)
content_matches = (rows_in_raw_not_bronze == 0 and rows_in_bronze_not_raw == 0)

if counts_match and content_matches:
    print("✅ SUCCESS: Data is 100% identical in both count and content!")
else:
    print("❌ ERROR: Mismatch detected! Check the numbers above.")
print("=" * 45)


In [0]:
from pyspark.sql import functions as F

# 1. Define the invalid condition
invalid_condition = (
    F.col("type").isNull()
    | ~F.col("type").isin("Movie", "TV Show")
    | F.col("show_id").isNull()
    | ~F.trim(F.col("show_id").cast("string")).rlike(r"^[0-9]+$")
    | F.col("_rescued_data").isNotNull()
)

# 2. Filter into Quarantine and Clean dataframes
quarantine_df = df_titles.filter(invalid_condition)
clean_df = df_titles.filter(~invalid_condition)

# 3. Optimize: Compute counts EXACTLY ONCE and store them in memory
total_bronze_rows = df_titles.count()
quarantine_rows = quarantine_df.count()
clean_rows = clean_rows = total_bronze_rows - quarantine_rows  # Math is faster than a 3rd scan!

# 4. Print results instantly using the cached Python variables
print("=" * 45)
print("🔍 BRONZE VALIDATION INTEGRITY CHECK")
print("=" * 45)
print(f"Bronze rows:                      {total_bronze_rows:,}")
print(f"Quarantined records:              {quarantine_rows:,}")
print(f"Records ready for transformation: {clean_rows:,}")
print("-" * 45)
print(f"Reconciliation passed:            {total_bronze_rows == (quarantine_rows + clean_rows)}")
print("=" * 45)


In [0]:
import pyspark.sql.functions as F

# Display with a dynamic reason column to triage ingestion issues
display(
    quarantine_df.withColumn(
        "quarantine_reason",
        F.when(F.col("_rescued_data").isNotNull(), "CSV Parsing Error / Corrupt Struct")
        .when(F.col("show_id").isNull(), "Missing show_id")
        .when(~F.trim(F.col("show_id").cast("string")).rlike(r"^[0-9]+$"), "Invalid show_id (Non-numeric)")
        .when(F.col("type").isNull(), "Missing type")
        .when(~F.col("type").isin("Movie", "TV Show"), "Unknown production type")
        .otherwise("Other schema mismatch")
    ).select(
        "show_id",
        "type",
        "title",
        "quarantine_reason",
        "_rescued_data",
        "_source_file"
    )
)


In [0]:
from pyspark.sql import functions as F

# 1. Define the expression to attempt casting the column to an integer
# If it's a string text like "abc", this expression automatically becomes NULL
cast_to_int = F.col("duration_minutes").cast("int")

# 2. Run a full Data Quality Profile summary
clean_df.select(
    # Check for empty / missing values
    F.sum(F.when(F.col("duration_minutes").isNull(), 1).otherwise(0)).alias("dq_null_count"),
    
    # Check for exact zeros
    F.sum(F.when(cast_to_int == 0, 1).otherwise(0)).alias("dq_zero_count"),
    
    # Check for negative entries
    F.sum(F.when(cast_to_int < 0, 1).otherwise(0)).alias("dq_negative_count"),
    
    # Check for non-integers (data was text/string that failed to convert to a number)
    F.sum(
        F.when(F.col("duration_minutes").isNotNull() & cast_to_int.isNull(), 1).otherwise(0)
    ).alias("dq_invalid_text_count")
).display()


**Investigate the 1,969 NULLs**

In [0]:
clean_df.filter(
    F.col("duration_minutes").isNull()
).groupBy("type").count().display()

All 1,969 NULLs are legitimate because TV Shows use duration_seasons instead.

**If future records contain zero, negative, or non-integer movie durations,quarantine those records.For TV Shows, NULL minutes remain valid.**

In [0]:
import pyspark.sql.functions as F

# 1. Define the exact invalid rules for Movies
# Rejects text/strings, zeros, negatives, and missing values for Movies only
is_invalid_movie_duration = (
    (F.col("type") == "Movie") & (
        F.col("duration_minutes").cast("int").isNull() |  # Catches text strings and True NULLs
        (F.col("duration_minutes").cast("int") <= 0)      # Catches zeros and negative numbers
    ) |
    ((F.col("type") == "TV Show") & F.col("duration_minutes").isNotNull())
)


# 2. Quarantine the bad records
quarantine_df = clean_df.filter(is_invalid_movie_duration)

# 3. Filter and Transform the clean records
# Casts valid movie strings (e.g. "90") to 90, while keeping TV Show NULLs intact
final_clean_df = (
    clean_df
    .filter(~is_invalid_movie_duration)
    .withColumn("duration_minutes", F.col("duration_minutes").cast("int"))
)


In [0]:
final_clean_df.display()

In [0]:
from pyspark.sql import functions as F

seasons_expr = F.col("duration_seasons").cast("int")

# 1. Single-pass summary of structural errors
clean_df.select(
    F.sum(F.when(F.col("duration_seasons").isNull(), 1).otherwise(0)).alias("dq_null_count"),
    F.sum(F.when(seasons_expr == 0, 1).otherwise(0)).alias("dq_zero_count"),
    F.sum(F.when(seasons_expr < 0, 1).otherwise(0)).alias("dq_negative_count"),
    F.sum(
        F.when(F.col("duration_seasons").isNotNull() & seasons_expr.isNull(), 1).otherwise(0)
    ).alias("dq_invalid_text_count")
).display()

# 2. Low-cardinality distribution check grouped by type
# Instantly exposes every distinct value (e.g., "1", "2", "NULL", or bad text) per content type
(
    clean_df
    .groupBy("type", "duration_seasons")
    .count()
    .orderBy("type", F.col("duration_seasons").cast("int").asc_nulls_first())
    .display()
)


In [0]:
from pyspark.sql import functions as F

# 1. Defensive rule for future batches: TV Shows must have seasons > 0, Movies must be NULL
is_invalid_seasons = (
    ((F.col("type") == "TV Show") & (
        F.col("duration_seasons").cast("int").isNull() |
        (F.col("duration_seasons").cast("int") <= 0)
    )) |
    ((F.col("type") == "Movie") & F.col("duration_seasons").isNotNull())
)

# 2. Quarantine any future bad records and cast valid ones to INT
quarantine_seasons_df = clean_df.filter(is_invalid_seasons)

clean_df = (
    clean_df
    .filter(~is_invalid_seasons)
    .withColumn("duration_seasons", F.col("duration_seasons").cast("int"))
)


In [0]:
clean_df.display()

 **check to inspect all distinct values, leading/trailing spaces, casing differences (like "movie" vs "Movie"), or unexpected categories.**

In [0]:
from pyspark.sql import functions as F

# 1. Inspect exact distinct values and counts (exposes whitespace, casing issues, or nulls)
clean_df.groupBy("type").count().orderBy(F.desc("count")).display()



Standardise & Quarantine (type)
Before rejecting rows, trim whitespace and normalise casing so minor formatting issues (like " movie " or "tv show") are fixed automatically instead of unnecessarily quarantined.


In [0]:
from pyspark.sql import functions as F

# 1. Standardise formatting (trim whitespace and apply Title Case)
clean_df = clean_df.withColumn(
    "type",
    F.when(F.trim(F.col("type")) == "", F.lit(None).cast("string"))
     .otherwise(F.initcap(F.trim(F.col("type"))))
)

# Note: F.initcap("tv show") produces "Tv Show", so map it explicitly to "TV Show"
clean_df = clean_df.withColumn(
    "type",
    F.when(F.lower(F.col("type")) == "tv show", F.lit("TV Show"))
     .when(F.lower(F.col("type")) == "movie", F.lit("Movie"))
     .otherwise(F.col("type"))
)

# 2. Quarantine any remaining records that are NULL or not in ("Movie", "TV Show")
is_invalid_type = (
    F.col("type").isNull() |
    ~F.col("type").isin("Movie", "TV Show")
)

quarantine_type_df = clean_df.filter(is_invalid_type)
clean_df = clean_df.filter(~is_invalid_type)


In [0]:
from pyspark.sql import functions as F

# Add an integer flag: 1 = Movie, 0 = TV Show
clean_df = clean_df.withColumn(
    "is_movie",
    F.when(F.col("type") == "Movie", F.lit(1))
     .otherwise(F.lit(0))
     .cast("int")
)


In [0]:
display(clean_df)

**check for missing titles, extra whitespace, or duplicate titles**

In [0]:
from pyspark.sql import functions as F

clean_df.select(
    F.sum(F.when(F.col("title").isNull() | (F.trim(F.col("title")) == ""), 1).otherwise(0)).alias("dq_missing_title"),
    F.sum(F.when(F.col("title") != F.trim(F.col("title")), 1).otherwise(0)).alias("dq_leading_trailing_spaces"),
    F.sum(F.when(F.col("title").rlike(r"\s{2,}"), 1).otherwise(0)).alias("dq_double_spaces"),
    (F.count("*") - F.countDistinct(F.lower(F.trim(F.col("title"))))).alias("dq_duplicate_titles")
).display()


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Window by case-insensitive title
w_title = Window.partitionBy(F.lower(F.col("title")))

# Window by exact content key (title + type + release_year)
w_exact = Window.partitionBy(F.lower(F.col("title")), F.col("type"), F.col("release_year"))

dup_audit_df = (
    clean_df
    .withColumn("same_title_count", F.count("*").over(w_title))
    .withColumn("exact_match_count", F.count("*").over(w_exact))
    .filter(F.col("same_title_count") > 1)
    .withColumn(
        "duplicate_category",
        F.when(F.col("exact_match_count") > 1, "TRUE DUPLICATE (Same Title + Type + Year)")
         .otherwise("DISTINCT RELEASE (Different Type or Year)")
    )
    .select(
        "duplicate_category",
        "show_id",
        "title",
        "type",
        "release_year",
        "date_added",
        "rating",
        "duration_minutes",
        "duration_seasons"
    )
    .orderBy("duplicate_category", F.lower(F.col("title")), "release_year")
)

# 1. Summary count by category
dup_audit_df.groupBy("duplicate_category").count().display()

# 2. Detailed side-by-side comparison of all duplicate rows
dup_audit_df.display()


In [0]:
from pyspark.sql import functions as F

# Hunt for weird colon patterns (like 23:59 or URLs) before writing a split rule
clean_df.filter(
    F.col("title").rlike(r":\S")  # Colon followed immediately by a non-space character
).select("title").distinct().display()


In [0]:
from pyspark.sql import functions as F

# 1. Standardise whitespace (trim outer spaces, collapse repeated internal spaces, blank -> NULL)
clean_df = clean_df.withColumn(
    "title",
    F.when(
        F.col("title").isNull() | (F.trim(F.col("title")) == ""),
        F.lit(None).cast("string")
    ).otherwise(
        F.trim(F.regexp_replace(F.col("title"), r"\s+", " "))
    )
)

# 2. Quarantine rows where title is NULL
is_invalid_title = F.col("title").isNull()
quarantine_title_df = clean_df.filter(is_invalid_title)
clean_df = clean_df.filter(~is_invalid_title)

# 3. Guardrail against true duplicates in future runs (preserves distinct releases across years/types)
clean_df = (
    clean_df
    .dropDuplicates(["show_id"])
    .dropDuplicates(["title", "type", "release_year"])
)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w_title = Window.partitionBy(F.lower(F.col("title")))

clean_df.withColumn("shared_cnt", F.count("*").over(w_title)).select(
    F.count("*").alias("total_clean_rows"),
    F.sum(F.when(F.col("title").isNull() | (F.trim(F.col("title")) == ""), 1).otherwise(0)).alias("val_null_titles"),
    F.sum(F.when(F.col("title") != F.trim(F.col("title")), 1).otherwise(0)).alias("val_outer_spaces"),
    F.sum(F.when(F.col("title").rlike(r"\s{2,}"), 1).otherwise(0)).alias("val_double_spaces"),
    F.sum(F.when(F.col("shared_cnt") > 1, 1).otherwise(0)).alias("preserved_shared_title_rows")
).display()


In [0]:
from pyspark.sql import functions as F

# Test parsing date_added (supports M/d/yyyy like '9/9/2019' and fallback 'MMMM d, yyyy' if future files vary)
parsed_date_test = F.coalesce(
    F.to_date(F.trim(F.col("date_added")), "M/d/yyyy"),
    F.to_date(F.trim(F.col("date_added")), "MMMM d, yyyy")
)
year_int = F.col("release_year").cast("int")

# 1. Single-pass audit summary
clean_df.select(
    # date_added checks
    F.sum(F.when(F.col("date_added").isNull() | (F.trim(F.col("date_added")) == ""), 1).otherwise(0)).alias("dq_null_date_added"),
    F.sum(F.when(F.col("date_added").isNotNull() & (F.trim(F.col("date_added")) != "") & parsed_date_test.isNull(), 1).otherwise(0)).alias("dq_unparseable_date_added"),
    
    # release_year checks
    F.sum(F.when(year_int.isNull(), 1).otherwise(0)).alias("dq_invalid_release_year"),
    F.sum(F.when((year_int < 1920) | (year_int > F.year(F.current_date()) + 1), 1).otherwise(0)).alias("dq_out_of_range_year"),
    
    # Cross-column logic check (added to Netflix before the release year)
    F.sum(F.when(F.year(parsed_date_test) < year_int, 1).otherwise(0)).alias("dq_added_before_release")
).display()

# 2. Inspect any unparseable date strings or cross-column anomalies
clean_df.filter(
    (F.col("date_added").isNotNull() & (F.trim(F.col("date_added")) != "") & parsed_date_test.isNull()) |
    (F.year(parsed_date_test) < year_int)
).select("show_id", "title", "type", "date_added", "release_year").display()


**catch missing values, unparseable date strings, unreasonable release years (e.g., before 1900 or in the future), and logical anomalies where a title was added to Netflix before its release year.**

In [0]:
from pyspark.sql import functions as F

# Test parsing date_added (supports M/d/yyyy like '9/9/2019' and fallback 'MMMM d, yyyy' if future files vary)
parsed_date_test = F.coalesce(
    F.to_date(F.trim(F.col("date_added")), "M/d/yyyy"),
    F.to_date(F.trim(F.col("date_added")), "MMMM d, yyyy")
)
year_int = F.col("release_year").cast("int")

# 1. Single-pass audit summary
clean_df.select(
    # date_added checks
    F.sum(F.when(F.col("date_added").isNull() | (F.trim(F.col("date_added")) == ""), 1).otherwise(0)).alias("dq_null_date_added"),
    F.sum(F.when(F.col("date_added").isNotNull() & (F.trim(F.col("date_added")) != "") & parsed_date_test.isNull(), 1).otherwise(0)).alias("dq_unparseable_date_added"),
    
    # release_year checks
    F.sum(F.when(year_int.isNull(), 1).otherwise(0)).alias("dq_invalid_release_year"),
    F.sum(F.when((year_int < 1920) | (year_int > F.year(F.current_date()) + 1), 1).otherwise(0)).alias("dq_out_of_range_year"),
    
    # Cross-column logic check (added to Netflix before the release year)
    F.sum(F.when(F.year(parsed_date_test) < year_int, 1).otherwise(0)).alias("dq_added_before_release")
).display()

# 2. Inspect any unparseable date strings or cross-column anomalies
clean_df.filter(
    (F.col("date_added").isNotNull() & (F.trim(F.col("date_added")) != "") & parsed_date_test.isNull()) |
    (F.year(parsed_date_test) < year_int)
).select("show_id", "title", "type", "date_added", "release_year").display()


In [0]:
from pyspark.sql import functions as F

parsed_date = F.coalesce(
    F.to_date(F.trim(F.col("date_added")), "M/d/yyyy"),
    F.to_date(F.trim(F.col("date_added")), "MMMM d, yyyy")
)

(
    clean_df
    .filter(F.year(parsed_date) < F.col("release_year").cast("int"))
    .withColumn("year_diff", F.year(parsed_date) - F.col("release_year").cast("int"))
    .select(
        "show_id",
        "type",
        "title",
        "date_added",
        "release_year",
        "year_diff",
        "duration_seasons",
        "duration_minutes"
    )
    .orderBy("type", "year_diff")
    .display()
)


In [0]:
from pyspark.sql import functions as F

# 1. Parse date_added to DateType and cast release_year to Int
parsed_date = F.coalesce(
    F.to_date(F.trim(F.col("date_added")), "M/d/yyyy"),
    F.to_date(F.trim(F.col("date_added")), "MMMM d, yyyy")
)
year_int = F.col("release_year").cast("int")

# 2. Quarantine only truly corrupt date strings or out-of-range release years
is_invalid_dates = (
    (F.col("date_added").isNotNull() & (F.trim(F.col("date_added")) != "") & parsed_date.isNull()) |
    year_int.isNull() |
    (year_int < 1920) |
    (year_int > F.year(F.current_date()) + 1)
)

quarantine_dates_df = clean_df.filter(is_invalid_dates)

# 3. Apply transformations and add non-blocking quality flag for the 9 source anomalies
clean_df = (
    clean_df
    .filter(~is_invalid_dates)
    .withColumn("date_added", parsed_date)
    .withColumn("release_year", year_int)
    .withColumn(
        "is_added_before_release",
        F.when(F.year(F.col("date_added")) < F.col("release_year"), F.lit(True))
         .otherwise(F.lit(False))
    )
)


In [0]:
display(clean_df)

In [0]:
from pyspark.sql import functions as F

# 1. Inspect the full low-cardinality distribution of rating values
(
    clean_df
    .groupBy("rating")
    .count()
    .orderBy(F.desc("count"))
    .display()
)

# 2. Check for nulls, outer spaces, or duration strings accidentally placed in rating
clean_df.select(
    F.sum(F.when(F.col("rating").isNull() | (F.trim(F.col("rating")) == ""), 1).otherwise(0)).alias("dq_null_or_blank_rating"),
    F.sum(F.when(F.col("rating") != F.trim(F.col("rating")), 1).otherwise(0)).alias("dq_outer_spaces"),
    F.sum(F.when(F.lower(F.col("rating")).rlike(r"(min|season)"), 1).otherwise(0)).alias("dq_leaked_durations")
).display()


Map null to "Not Rated"

In [0]:
from pyspark.sql import functions as F

# 1. Standardise casing, trim whitespace, and normalise any accidental space after "TV" (e.g., "TV MA" -> "TV-MA")
clean_df = clean_df.withColumn(
    "rating",
    F.when(F.col("rating").isNull() | (F.trim(F.col("rating")) == ""), F.lit("Not Rated"))
     .otherwise(
         F.regexp_replace(
             F.upper(F.trim(F.col("rating"))),
             r"^TV\s+",
             "TV-"
         )
     )
)

# 2. Defensive allowlist for future incremental loads (catches leaked durations like "74 min")
valid_ratings = [
    "TV-MA", "TV-14", "TV-PG", "TV-Y", "TV-Y7", "TV-Y7-FV", "TV-G",
    "R", "PG-13", "PG", "G", "NC-17", "NR", "UR", "Not Rated"
]

is_invalid_rating = ~F.col("rating").isin(valid_ratings)

quarantine_rating_df = clean_df.filter(is_invalid_rating)
clean_df = clean_df.filter(~is_invalid_rating)

# 3. Post-transformation validation check
clean_df.groupBy("rating").count().orderBy(F.desc("count")).display()


In [0]:
from pyspark.sql import functions as F

# 1. Single-pass summary of free-text quality issues
clean_df.select(
    F.sum(F.when(F.col("description").isNull() | (F.trim(F.col("description")) == ""), 1).otherwise(0)).alias("dq_missing_desc"),
    F.sum(F.when(F.col("description") != F.trim(F.col("description")), 1).otherwise(0)).alias("dq_outer_spaces"),
    F.sum(F.when(F.col("description").rlike(r"[\r\n]+"), 1).otherwise(0)).alias("dq_embedded_newlines"),
    F.sum(F.when(F.col("description").rlike(r"\s{2,}"), 1).otherwise(0)).alias("dq_double_spaces"),
    F.min(F.length(F.trim(F.col("description")))).alias("min_char_length"),
    F.max(F.length(F.trim(F.col("description")))).alias("max_char_length")
).display()

# 2. Inspect the 10 shortest descriptions to spot any truncated or broken text
(
    clean_df
    .filter(F.col("description").isNotNull())
    .select("show_id", "title", "description", F.length("description").alias("char_len"))
    .orderBy("char_len")
    .limit(10)
    .display()
)


In [0]:
from pyspark.sql import functions as F

# Collapse newlines/tabs/multiple spaces into a single space, trim outer spaces, and map empty strings to NULL
clean_df = clean_df.withColumn(
    "description",
    F.when(
        F.col("description").isNull() | (F.trim(F.col("description")) == ""),
        F.lit(None).cast("string")
    ).otherwise(
        F.trim(F.regexp_replace(F.col("description"), r"\s+", " "))
    )
)


In [0]:
from pyspark.sql import functions as F

clean_df.select(
    F.count("*").alias("total_clean_rows"),
    F.sum(F.when(F.col("description").isNull(), 1).otherwise(0)).alias("remaining_null_desc"),
    F.sum(F.when(F.col("description") != F.trim(F.col("description")), 1).otherwise(0)).alias("val_outer_spaces"),
    F.sum(F.when(F.col("description").rlike(r"[\r\n]+"), 1).otherwise(0)).alias("val_embedded_newlines"),
    F.sum(F.when(F.col("description").rlike(r"\s{2,}"), 1).otherwise(0)).alias("val_double_spaces")
).display()


**verify there are no missing IDs, non-numeric characters, zero/negative numbers, or duplicate show_id values.**

In [0]:
from pyspark.sql import functions as F

id_int = F.col("show_id").cast("int")

clean_df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("show_id").isNull() | (F.trim(F.col("show_id").cast("string")) == ""), 1).otherwise(0)).alias("dq_null_show_id"),
    F.sum(F.when(F.col("show_id").isNotNull() & id_int.isNull(), 1).otherwise(0)).alias("dq_non_numeric_show_id"),
    F.sum(F.when(id_int <= 0, 1).otherwise(0)).alias("dq_zero_or_negative_id"),
    (F.count("*") - F.countDistinct("show_id")).alias("dq_duplicate_show_ids")
).display()


In [0]:
from pyspark.sql import functions as F

# 1. Trim whitespace and cast cleanly to STRING (preserves leading zeros like "0012345")
id_str = F.trim(F.col("show_id").cast("string"))

# 2. Quarantine any row where show_id is NULL, empty, or contains non-digit characters
is_invalid_show_id = (
    id_str.isNull() |
    (id_str == "") |
    ~id_str.rlike(r"^[0-9]+$")
)

quarantine_show_id_df = clean_df.filter(is_invalid_show_id)

# 3. Keep valid IDs as STRING and enforce primary key uniqueness
clean_df = (
    clean_df
    .filter(~is_invalid_show_id)
    .withColumn("show_id", id_str)
    .dropDuplicates(["show_id"])
)


In [0]:
from pyspark.sql import functions as F

clean_df.select(
    F.count("*").alias("total_clean_rows"),
    F.countDistinct("show_id").alias("distinct_show_ids"),
    F.sum(F.when(F.col("show_id").isNull() | ~F.col("show_id").rlike(r"^[0-9]+$"), 1).otherwise(0)).alias("val_invalid_ids"),
    F.min(F.length("show_id")).alias("min_id_length"),
    F.max(F.length("show_id")).alias("max_id_length")
).display()
